In [ ]:
import sys, platform
print("PYTHON", sys.version)
print("PLATFORM", platform.platform())
from pathlib import Path
print("CONTENT_READY", Path("/content").is_dir())


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
for p in [Path("/content/drive/MyDrive/BenchPress_FromScratch/models/bench_press_anomaly_trained.joblib"), Path("/content/drive/MyDrive/Exercise_AI_Models/Chest")]:
    print(str(p), "EXISTS", p.exists())


In [ ]:
from pathlib import Path
exec(compile(Path("/content/chest_common_cell.py").read_text(), "chest_common_cell.py", "exec"))
import pandas as pd
results = [train_one(key) for key in EXERCISES]
print(pd.DataFrame(results).to_string(index=False))
print("Real exercise models trained:", sum(r["status"].startswith("TRAINED") for r in results))


In [ ]:
import mediapipe as mp
print("MEDIAPIPE", mp.__version__)
detector = make_landmarker()
try:
    assert pose_feature(np.zeros((256,256,3),dtype=np.uint8),detector) is None
    print("PASS: Tasks PoseLandmarker loaded; blank non-human input rejected")
finally:
    detector.close()
print("This is an API smoke test, not exercise training or accuracy validation.")


In [ ]:
# MIT OCW PE.720 (Halston Taylor, Spring 2006). CC BY-NC-SA.
# User approved noncommercial experiment; these are demonstrations, NOT verified GOOD/BAD labels.
import requests, hashlib, json, cv2
from pathlib import Path
mit_dir=Path('/content/mit_ocw_chest'); mit_dir.mkdir(exist_ok=True)
mit_names=['decline_bench_press','incline_bench_press_first_view','incline_bench_press_second_view','cable_crossover','chest_flies']
mit_records=[]
for name in mit_names:
    url='https://www.archive.org/download/MITPE.720S06/'+name+'-220k.mp4'
    try:
        r=requests.get(url,timeout=45); r.raise_for_status()
        assert len(r.content)<50_000_000
        p=mit_dir/(name+'.mp4'); p.write_bytes(r.content)
        cap=cv2.VideoCapture(str(p)); n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps=cap.get(cv2.CAP_PROP_FPS); cap.release()
        assert n>0, 'not decodable video'
        row=dict(name=name,path=str(p),url=url,bytes=p.stat().st_size,sha256=hashlib.sha256(r.content).hexdigest(),frames=n,fps=fps,quality_label='UNVERIFIED_DEMONSTRATION',source_group='MIT_PE720_2006',subject_id='UNKNOWN',license='CC BY-NC-SA; see MIT OCW terms',source='https://ocw.mit.edu/courses/pe-720-weight-training-spring-2006/pages/video/')
        mit_records.append(row); print('ACQUIRED',name,n,'frames',round(n/fps,2),'seconds')
    except Exception as e: print('FAILED',name,type(e).__name__,str(e)[:200])
(mit_dir/'acquisition.json').write_text(json.dumps(mit_records,indent=2))
print('Actual unique video hashes:',len(set(r['sha256'] for r in mit_records)))
print('Verified quality labels: 0; independent subjects unknown. No accuracy claim.')


In [ ]:
import matplotlib.pyplot as plt
fig,axs=plt.subplots(5,3,figsize=(12,10))
for row,rec in enumerate(mit_records):
 cap=cv2.VideoCapture(rec['path'])
 for col,fraction in enumerate([.25,.5,.75]):
  cap.set(cv2.CAP_PROP_POS_FRAMES,int(rec['frames']*fraction)); ok,frame=cap.read()
  if ok: axs[row,col].imshow(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
  axs[row,col].axis('off'); axs[row,col].set_title(rec['name'],fontsize=8)
 cap.release()
plt.tight_layout(); plt.show()


In [ ]:
# Feasibility audit. OpenCV BGR explicitly converted to RGB.
detector=make_landmarker(); mit_audit=[]
try:
 for rec in mit_records:
  cap=cv2.VideoCapture(rec['path']); features=[]; sampled=0
  for idx in np.linspace(0,rec['frames']-1,24,dtype=int):
   cap.set(cv2.CAP_PROP_POS_FRAMES,int(idx)); ok,frame=cap.read()
   if not ok: continue
   sampled+=1; feat=pose_feature(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB),detector)
   if feat is not None: features.append(feat)
  cap.release()
  row=dict(name=rec['name'],sampled_frames=sampled,accepted_pose_frames=len(features),unique_media=1,independent_subjects='UNKNOWN',quality_label='UNVERIFIED_DEMONSTRATION',training_status='NOT_TRAINED_INSUFFICIENT_INDEPENDENT_MEDIA_AND_QUALITY_LABELS')
  mit_audit.append(row); print(json.dumps(row))
finally: detector.close()
(mit_dir/'pose_audit.json').write_text(json.dumps(mit_audit,indent=2))
print('No form-quality model trained. No external accuracy measurable.')
